# Eksperimen CNN Malaria: Pilot Learning Rate + 12 Konfigurasi

Notebook ini mengimplementasikan protokol penelitian **“Pengaruh Batch Normalization, Dropout, dan Learning Rate Scheduling terhadap Kinerja CNN dalam Klasifikasi Infeksi Malaria”**.

**Alur otomatis saat `Run All`:**
1. Menemukan/mengunduh dataset, mengaudit kelas dan duplikat MD5.
2. Membentuk `slide_id` dan `group_id`, lalu membuat satu `split_manifest.csv` dengan `StratifiedGroupKFold` (5 fold train, 1 validation, 1 test).
3. Menjalankan pilot baseline A untuk LR `3e-4` dan `1e-3` (15 epoch masing-masing), memilih LR dengan *minimum validation loss* yang lebih kecil, lalu menyimpannya.
4. Melatih konfigurasi A–L sesuai fase dan seed pada protokol (maksimum 60 pelatihan, masing-masing 40 epoch).
5. Setelah seluruh pelatihan yang dijadwalkan selesai, membuka test set satu kali per run untuk evaluasi final.
6. Membuat tabel ringkasan, perbandingan berpasangan, ANOVA tiga arah (hanya jika desain 12 konfigurasi lengkap dan seimbang), kurva, dan contoh FP/FN.

**Default notebook:** 12 konfigurasi × 5 seed (60 pelatihan) + 2 pelatihan pilot. Ini pekerjaan komputasi besar; gunakan GPU (di Kaggle Notebook pilih Accelerator → GPU). Bila notebook terputus, jalankan lagi dari atas: manifest, pilot, dan run yang sudah selesai akan digunakan kembali, sedangkan run yang belum lengkap akan diulang.

> **Catatan akses dataset:** bila berjalan di Kaggle Notebook, tambahkan dataset ini melalui *Add Input*. Di komputer lokal/Colab, notebook memakai Kaggle CLI dan memerlukan `kaggle.json` atau variabel lingkungan `KAGGLE_USERNAME` dan `KAGGLE_KEY`. Panduan ada di sel persiapan.

**Konsistensi protokol:** input default 128×128 RGB, tanpa augmentasi, float32, batch 64, Adam, BCE, 40 epoch, tanpa early stopping, checkpoint berdasarkan validation loss, threshold 0,5. Ukuran gambar dapat diubah menjadi 64 hanya bila seluruh eksperimen memang akan dijalankan pada 64×64; jangan campur ukuran input antar-run.

## 0. Persiapan lingkungan

Sel ini memasang pustaka yang belum tersedia. TensorFlow tidak dipaksa dipasang ulang bila sudah terinstal. Pada Windows native, kompatibilitas GPU TensorFlow bergantung pada versi dan lingkungan; WSL2/Linux atau Kaggle Notebook umumnya menjadi pilihan yang lebih sederhana untuk GPU.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "statsmodels": "statsmodels",
    "kaggle": "kaggle",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec("tensorflow") is None:
    missing.append("tensorflow")
if missing:
    print("Memasang pustaka:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Semua pustaka utama sudah tersedia.")

Memasang pustaka: statsmodels, kaggle, tensorflow


## 1. Konfigurasi eksperimen

Default di bawah menjalankan **seluruh 12 konfigurasi dan lima seed** sesuai protokol. `MAX_PHASE_TO_RUN=4` berarti fase 1–4 dijalankan penuh. Jangan mengubah hyperparameter di tengah eksperimen; jika ukuran input diubah menjadi 64, gunakan nilai itu untuk semua konfigurasi dan dokumentasikan sebagai deviasi protokol.

In [2]:
from pathlib import Path
import os, re, json, time, hashlib, random, shutil, gc, platform, sys, subprocess, warnings
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt

# Lokasi kerja: /kaggle/working pada Kaggle, atau folder kerja notebook lokal.
IS_KAGGLE = Path("/kaggle/working").exists() and Path("/kaggle/input").exists()
BASE_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUTPUT_DIR = BASE_DIR / "malaria_cnn_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "iarunava/cell-images-for-detecting-malaria"
DATASET_HINT = None  # Opsional: isi path folder yang memuat folder Parasitized dan Uninfected.

IMAGE_SIZE = 128             # Protokol: 128; ubah ke 64 hanya bila dipakai untuk SEMUA run.
BATCH_SIZE = 64
EPOCHS = 40
PILOT_EPOCHS = 15
PILOT_LR_CANDIDATES = [3e-4, 1e-3]
SEEDS = [1, 2, 3, 4, 5]
MAX_PHASE_TO_RUN = 4          # 4 = semua 12 konfigurasi; 2 = A-H; 1 = A-F.
FORCE_REBUILD_SPLIT = False
FORCE_RERUN_PILOT = False
FORCE_RERUN_COMPLETED_RUNS = False

EXPECTED_PER_CLASS = 13779
EXPECTED_TOTAL = 27558
SPLIT_RANDOM_STATE = 42          # dipakai langsung bila AUTO_SEARCH_SPLIT_SEED = False
# Pencarian seed split otomatis (hanya memakai label & group_id; tidak memakai hasil model apa pun).
# Dijalankan saat split dibuat dari awal (tidak ada split_manifest.csv, atau FORCE_REBUILD_SPLIT = True).
AUTO_SEARCH_SPLIT_SEED = True
SPLIT_SEED_CANDIDATES = range(0, 1000)
SPLIT_CLASS_TOL = 0.02           # penyimpangan proporsi parasitized dari 0,5 per split (sama dengan assert)
SPLIT_SIZE_TOL = 0.01            # penyimpangan porsi citra per split dari target 5/7, 1/7, 1/7
CLASS_NAMES = {0: "Uninfected", 1: "Parasitized"}
IMG_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# Parameter tetap dari protokol.
INITIAL_LR = None  # Diisi otomatis dari pilot.
MIN_LR = 1e-6
DROPOUT_CONV = 0.25
DROPOUT_DENSE = 0.50
THRESHOLD = 0.50

for sub in ["pilot", "runs", "tables", "plots", "error_examples"]:
    (OUTPUT_DIR / sub).mkdir(exist_ok=True)

print("Runtime:", "Kaggle Notebook" if IS_KAGGLE else "Lokal/Colab")
print("Output:", OUTPUT_DIR.resolve())
print("Input:", f"{IMAGE_SIZE}x{IMAGE_SIZE} RGB | batch={BATCH_SIZE} | epoch={EPOCHS}")
print("Fase maksimal:", MAX_PHASE_TO_RUN, "(4 berarti A-L dan seed 1-5)")

Runtime: Lokal/Colab
Output: D:\KULIAH FADHIL\SEMESTER 7\DEEP LEARNING\UTS\malaria_cnn_experiment
Input: 128x128 RGB | batch=64 | epoch=40
Fase maksimal: 4 (4 berarti A-L dan seed 1-5)


### 1A. Dataset dan autentikasi Kaggle

Notebook terlebih dahulu mencari folder dataset yang sudah tersedia (misalnya di `/kaggle/input`). Jika tidak ditemukan, notebook mencoba mengunduh dengan perintah resmi Kaggle CLI. Untuk penggunaan lokal, buat token API dari halaman akun Kaggle (`Settings` → `API` → `Create New Token`) dan letakkan `kaggle.json` di `~/.kaggle/kaggle.json` (Linux/WSL) atau `%USERPROFILE%\.kaggle\kaggle.json` (Windows). Jangan unggah atau membagikan token API di notebook publik.

In [3]:
def locate_class_directories(search_roots):
    """Cari satu pasangan folder Parasitized/Uninfected pada parent yang sama."""
    checked = set()
    for root in search_roots:
        root = Path(root).expanduser()
        if not root.exists():
            continue
        try:
            for p in root.rglob("Parasitized"):
                if not p.is_dir():
                    continue
                parent = p.parent.resolve()
                key = str(parent).lower()
                if key in checked:
                    continue
                checked.add(key)
                uninfected = parent / "Uninfected"
                if uninfected.is_dir():
                    return parent, p.resolve(), uninfected.resolve()
        except (PermissionError, OSError):
            continue
    return None

search_roots = []
if DATASET_HINT:
    search_roots.append(DATASET_HINT)
if IS_KAGGLE:
    search_roots.append("/kaggle/input")
search_roots.extend([OUTPUT_DIR / "download", BASE_DIR / "data", BASE_DIR])
found = locate_class_directories(search_roots)

if found is None:
    download_dir = OUTPUT_DIR / "download"
    download_dir.mkdir(parents=True, exist_ok=True)
    print("Dataset belum ditemukan. Mencoba Kaggle CLI...")
    kaggle_json_candidates = [
        Path.home() / ".kaggle" / "kaggle.json",
        Path(os.environ.get("KAGGLE_CONFIG_DIR", "~/.kaggle")).expanduser() / "kaggle.json",
    ]
    has_env_credentials = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_file_credentials = any(p.exists() for p in kaggle_json_candidates)
    if not (has_env_credentials or has_file_credentials):
        raise RuntimeError(
            "Dataset belum tersedia dan kredensial Kaggle tidak ditemukan.\n"
            "Tambahkan dataset lewat Add Input (Kaggle Notebook), atau siapkan kaggle.json "
            "di ~/.kaggle/kaggle.json (Linux/WSL) / %USERPROFILE%\\.kaggle\\kaggle.json (Windows), "
            "atau set KAGGLE_USERNAME dan KAGGLE_KEY. Lalu jalankan ulang notebook."
        )
    cmd = [sys.executable, "-m", "kaggle", "datasets", "download", "-d", DATASET_ID,
           "-p", str(download_dir), "--unzip"]
    subprocess.check_call(cmd)
    found = locate_class_directories([download_dir])

if found is None:
    raise FileNotFoundError(
        "Folder kelas tidak ditemukan. Struktur yang diharapkan adalah "
        ".../cell_images/Parasitized dan .../cell_images/Uninfected. "
        "Periksa DATASET_HINT atau struktur hasil ekstraksi."
    )

DATASET_ROOT, PARASITIZED_DIR, UNINFECTED_DIR = found
print("Dataset root:", DATASET_ROOT)
print("Parasitized:", PARASITIZED_DIR)
print("Uninfected:", UNINFECTED_DIR)

Dataset belum ditemukan. Mencoba Kaggle CLI...


RuntimeError: Dataset belum tersedia dan kredensial Kaggle tidak ditemukan.
Tambahkan dataset lewat Add Input (Kaggle Notebook), atau siapkan kaggle.json di ~/.kaggle/kaggle.json (Linux/WSL) / %USERPROFILE%\.kaggle\kaggle.json (Windows), atau set KAGGLE_USERNAME dan KAGGLE_KEY. Lalu jalankan ulang notebook.

## 2. Audit data dan pembentukan manifest split

- Hanya file gambar dari dua folder kelas yang digunakan; file non-gambar seperti `Thumbs.db` diabaikan dan jumlahnya dicatat.
- Audit menghitung MD5, duplikat persis, label yang bertentangan, ID slide, dan ID grup pasien.
- `slide_id` diambil dari nama file sebelum `_IMG_`. Untuk slide terinfeksi, regex nomor pasien menggabungkan dua slide P8 ke satu grup (`pasien_p8`); slide sehat memakai `slide_id` sebagai `group_id`.
- `StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)` digunakan untuk menetapkan satu fold test, satu fold validation, dan lima fold train.
- Manifest disimpan sekali dan dipakai ulang untuk seluruh konfigurasi/seed. Test tidak dibaca di sini; baru dibuat sebagai pipeline setelah seluruh training selesai.

In [ ]:
import tensorflow as tf
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score)

# Float32 sesuai protokol. Tidak menggunakan mixed precision.
try:
    tf.keras.mixed_precision.set_global_policy("float32")
except Exception:
    pass

# GPU memory growth, bila GPU tersedia.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("Peringatan pengaturan memory growth:", exc)
try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISM_ENABLED = True
except Exception as exc:
    DETERMINISM_ENABLED = False
    print("Operasi deterministik tidak dapat diaktifkan:", exc)

print("TensorFlow:", tf.__version__)
print("GPU:", gpus if gpus else "Tidak terdeteksi — training akan berjalan di CPU")
print("Deterministic ops:", DETERMINISM_ENABLED)

records = []
ignored_non_images = {"Parasitized": 0, "Uninfected": 0}
for folder, label in [(PARASITIZED_DIR, 1), (UNINFECTED_DIR, 0)]:
    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMG_EXTENSIONS:
            ignored_non_images[folder.name] += 1
            continue
        # Cek file benar-benar dapat dibaca; tidak mengubah berkas sumber.
        try:
            with Image.open(path) as im:
                im.verify()
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            raise RuntimeError(f"File gambar rusak/tidak terbaca: {path} ({exc})")
        records.append({"path": str(path.resolve()), "label": label, "class_name": CLASS_NAMES[label]})

df = pd.DataFrame(records).sort_values("path").reset_index(drop=True)
class_counts = df["label"].value_counts().to_dict()
print("Jumlah kelas:", {CLASS_NAMES[k]: int(class_counts.get(k, 0)) for k in [0, 1]})
print("File non-gambar diabaikan:", ignored_non_images)

if len(df) != EXPECTED_TOTAL or class_counts.get(1, 0) != EXPECTED_PER_CLASS or class_counts.get(0, 0) != EXPECTED_PER_CLASS:
    raise AssertionError(
        f"Audit kelas tidak sesuai protokol. Ditemukan total={len(df)}, "
        f"parasitized={class_counts.get(1,0)}, uninfected={class_counts.get(0,0)}; "
        f"target={EXPECTED_TOTAL} dan {EXPECTED_PER_CLASS} per kelas. "
        "Jangan lanjutkan sebelum memastikan versi/struktur dataset."
    )

# ID slide diambil persis dari bagian nama file sebelum _IMG_.
def get_slide_id(file_path):
    stem = Path(file_path).stem
    if "_IMG_" not in stem:
        raise ValueError(f"Nama file tidak memuat _IMG_: {Path(file_path).name}")
    return stem.split("_IMG_")[0].lower()

def get_group_id(slide_id):
    match = re.match(r"^c\d+[a-z]*?p(\d+)", slide_id, flags=re.IGNORECASE)
    if match:
        return f"pasien_p{int(match.group(1))}"
    return slide_id

df["slide_id"] = df["path"].map(get_slide_id)
df["group_id"] = df["slide_id"].map(get_group_id)

# MD5 audit untuk duplikat persis.
def md5_file(file_path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

md5_values = []
for i, p in enumerate(df["path"], start=1):
    md5_values.append(md5_file(p))
    if i % 5000 == 0:
        print(f"MD5: {i:,}/{len(df):,} file")
df["md5"] = md5_values
dataset_fingerprint = hashlib.sha256("\n".join((df.path + "|" + df.md5).tolist()).encode("utf-8")).hexdigest()
n_duplicate_extras = int(df.duplicated("md5", keep="first").sum())
unique_hashes = int(df["md5"].nunique())
label_conflicts = int((df.groupby("md5")["label"].nunique() > 1).sum())
print(f"Hash unik: {unique_hashes:,}; duplikat persis tambahan: {n_duplicate_extras:,}; hash dengan label konflik: {label_conflicts}")
if label_conflicts:
    raise AssertionError("Ada gambar identik dengan label berbeda. Periksa audit MD5 sebelum eksperimen.")

# Pastikan pola slide/group sesuai audit pada protokol.
EXPECTED_SLIDES = 201                  # slide_id unik
EXPECTED_PARASITIZED_SLIDES = 151      # slide yang memuat sel parasitized
EXPECTED_UNINFECTED_ONLY_SLIDES = 50   # slide yang hanya berisi sel uninfected
EXPECTED_INFECTED_PATIENT_GROUPS = 150 # P1-P150 (P8 punya dua slide)
EXPECTED_GROUPS = 200                  # 150 pasien terinfeksi + 50 slide sehat

parasitized_slides = set(df.loc[df.label == 1, "slide_id"])
uninfected_only_slides = set(df["slide_id"]) - parasitized_slides
infected_groups = set(df.loc[df.slide_id.isin(parasitized_slides), "group_id"])
healthy_groups = set(df.loc[df.slide_id.isin(uninfected_only_slides), "group_id"])
p8_slides = set(df.loc[df.group_id == "pasien_p8", "slide_id"])

print("Slide unik:", df["slide_id"].nunique(), "| Group unik:", df["group_id"].nunique())
print("Slide dengan sel parasitized:", len(parasitized_slides))
print("Slide hanya berisi sel uninfected:", len(uninfected_only_slides))
print("Group pasien terinfeksi:", len(infected_groups), "| Group slide sehat:", len(healthy_groups))
print("Slide pada group pasien_p8:", sorted(p8_slides))

assert df["slide_id"].nunique() == EXPECTED_SLIDES, \
    f"Jumlah slide_id unik {df['slide_id'].nunique()} != {EXPECTED_SLIDES} (protokol)."
assert len(parasitized_slides) == EXPECTED_PARASITIZED_SLIDES, \
    f"Slide dengan sel parasitized {len(parasitized_slides)} != {EXPECTED_PARASITIZED_SLIDES} (protokol)."
assert len(uninfected_only_slides) == EXPECTED_UNINFECTED_ONLY_SLIDES, \
    f"Slide hanya-uninfected {len(uninfected_only_slides)} != {EXPECTED_UNINFECTED_ONLY_SLIDES} (protokol)."
assert len(infected_groups) == EXPECTED_INFECTED_PATIENT_GROUPS, \
    f"Group pasien terinfeksi {len(infected_groups)} != {EXPECTED_INFECTED_PATIENT_GROUPS} (protokol)."
assert df["group_id"].nunique() == EXPECTED_GROUPS, \
    f"Jumlah group_id unik {df['group_id'].nunique()} != {EXPECTED_GROUPS} (protokol)."
# Slide sehat harus memakai slide_id sebagai group_id dan tidak boleh tergabung ke group pasien terinfeksi.
assert healthy_groups == uninfected_only_slides, \
    "group_id slide sehat tidak sama dengan slide_id-nya; periksa regex get_group_id."
assert not (infected_groups & healthy_groups), \
    f"Group slide sehat bertabrakan dengan group pasien terinfeksi: {sorted(infected_groups & healthy_groups)[:5]}"
assert len(p8_slides) == 2, f"pasien_p8 seharusnya memuat 2 slide, ditemukan {len(p8_slides)}."
print("Audit slide/group sesuai protokol.")

# Simpan audit lengkap.
aud = df[["path", "label", "class_name", "slide_id", "group_id", "md5"]].copy()
aud.to_csv(OUTPUT_DIR / "audit_data.csv", index=False)
audit_summary = {
    "total_images": int(len(df)),
    "parasitized": int((df.label == 1).sum()),
    "uninfected": int((df.label == 0).sum()),
    "unique_md5": unique_hashes,
    "duplicate_exact_extra_files": n_duplicate_extras,
    "md5_label_conflict_groups": label_conflicts,
    "unique_slides": int(df.slide_id.nunique()),
    "unique_groups": int(df.group_id.nunique()),
    "ignored_non_images": ignored_non_images,
    "dataset_fingerprint": dataset_fingerprint,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
(OUTPUT_DIR / "audit_summary.json").write_text(json.dumps(audit_summary, indent=2), encoding="utf-8")

# Pencarian seed split otomatis: hanya memakai label & group_id (komposisi data), bukan hasil model.
def search_split_seed(y, g):
    import sklearn
    targets = {"train": 5 / 7, "validation": 1 / 7, "test": 1 / 7}
    t0 = time.time()
    rows = []
    for seed in SPLIT_SEED_CANDIDATES:
        sp = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
        fold = np.full(len(y), -1, dtype=np.int32)
        for k, (_, idx) in enumerate(sp.split(np.zeros(len(y)), y, groups=g)):
            fold[idx] = k
        split = np.where(fold == 0, "test", np.where(fold == 1, "validation", "train"))
        row = {"seed": int(seed)}
        for name in targets:
            m = split == name
            row[f"{name}_frac"] = float(y[m].mean())
            row[f"{name}_size"] = float(m.mean())
        row["class_dev"] = max(abs(row[f"{n}_frac"] - 0.5) for n in targets)
        row["size_dev"] = max(abs(row[f"{n}_size"] - t) for n, t in targets.items())
        row["feasible"] = bool(row["class_dev"] <= SPLIT_CLASS_TOL and row["size_dev"] <= SPLIT_SIZE_TOL)
        rows.append(row)
    res = pd.DataFrame(rows)
    res.to_csv(OUTPUT_DIR / "tables" / "split_seed_search.csv", index=False)
    feasible = res[res.feasible].sort_values(["class_dev", "size_dev", "seed"])
    print(f"Pencarian seed split: {len(res)} seed dicoba, {len(feasible)} lolos kriteria, {time.time() - t0:.0f} dtk")
    if feasible.empty:
        raise RuntimeError("Tidak ada seed yang memenuhi kriteria split. Perluas SPLIT_SEED_CANDIDATES "
                           "(jangan melonggarkan toleransi tanpa mencatatnya di laporan).")
    best = feasible.iloc[0]
    selection = {
        "criteria": {"class_tol": SPLIT_CLASS_TOL, "size_tol": SPLIT_SIZE_TOL,
                     "ranking": "class_dev, lalu size_dev, lalu seed terkecil",
                     "uses_model_results": False},
        "seed_candidates": [int(min(SPLIT_SEED_CANDIDATES)), int(max(SPLIT_SEED_CANDIDATES))],
        "seeds_tried": int(len(res)),
        "seeds_feasible": int(len(feasible)),
        "selected_seed": int(best.seed),
        "sklearn_version": sklearn.__version__,
        "selected_metrics": {k: float(best[k]) for k in
                             ["train_frac", "validation_frac", "test_frac",
                              "train_size", "validation_size", "test_size"]},
    }
    (OUTPUT_DIR / "split_seed_selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
    print(f"Seed split terpilih: {int(best.seed)} (disimpan di split_seed_selection.json)")
    return int(best.seed)

manifest_path = OUTPUT_DIR / "split_manifest.csv"
if manifest_path.exists() and not FORCE_REBUILD_SPLIT:
    manifest = pd.read_csv(manifest_path)
    required_cols = {"path", "label", "slide_id", "group_id", "split"}
    if not required_cols.issubset(manifest.columns):
        raise ValueError("split_manifest.csv yang ada tidak memiliki kolom sesuai protokol.")
    current_paths = set(df.path)
    manifest_paths = set(manifest.path.astype(str))
    if current_paths != manifest_paths:
        raise ValueError("Dataset/path berbeda dari split_manifest.csv lama. Hapus/arsipkan output lama atau set FORCE_REBUILD_SPLIT=True.")
    manifest = manifest[["path", "label", "slide_id", "group_id", "split"]].copy()
    print("Memakai split_manifest.csv yang sudah ada.")
else:
    if AUTO_SEARCH_SPLIT_SEED:
        SPLIT_RANDOM_STATE = search_split_seed(df.label.values, df.group_id.values)
    splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SPLIT_RANDOM_STATE)
    fold_assignment = np.full(len(df), -1, dtype=np.int32)
    for fold_no, (_, fold_test_idx) in enumerate(splitter.split(np.zeros(len(df)), df.label.values, groups=df.group_id.values)):
        fold_assignment[fold_test_idx] = fold_no
    if np.any(fold_assignment < 0):
        raise RuntimeError("Sebagian citra tidak memperoleh fold.")
    split_name = np.where(fold_assignment == 0, "test", np.where(fold_assignment == 1, "validation", "train"))
    manifest = df[["path", "label", "slide_id", "group_id"]].copy()
    manifest["split"] = split_name
    manifest.to_csv(manifest_path, index=False)
    print("Manifest split baru disimpan.")

# Verifikasi manifest, proporsi kelas, dan pemisahan grup.
manifest["label"] = manifest.label.astype(int)
sets = {s: manifest.loc[manifest.split == s].copy() for s in ["train", "validation", "test"]}
if any(len(sets[s]) == 0 for s in sets):
    raise AssertionError("Ada split kosong.")
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = set(sets[a].group_id) & set(sets[b].group_id)
    assert not overlap, f"Group leakage antara {a} dan {b}: {list(overlap)[:5]}"

# Duplikat MD5 tidak boleh melintasi split berbeda. Duplikat dalam split yang sama hanya dicatat,
# tidak dihapus, karena protokol menginstruksikan audit MD5 dan mempertahankan katalog dataset.
path_to_md5 = df.set_index("path")["md5"]
manifest["md5"] = manifest.path.map(path_to_md5)
md5_split_counts = manifest.groupby("md5").split.nunique()
cross_split_dups = int((md5_split_counts > 1).sum())
if cross_split_dups:
    raise AssertionError(f"Ditemukan {cross_split_dups} hash duplikat yang melintasi split. Ini berpotensi leakage; perlu penanganan sebelum training.")

split_rows = []
for split_name, part in sets.items():
    ratio = float(part.label.mean())
    row = {
        "split": split_name,
        "images": int(len(part)),
        "groups": int(part.group_id.nunique()),
        "slides": int(part.slide_id.nunique()),
        "parasitized": int((part.label == 1).sum()),
        "uninfected": int((part.label == 0).sum()),
        "parasitized_fraction": ratio,
    }
    split_rows.append(row)
    print(row)
    if abs(ratio - 0.5) > 0.02:
        raise AssertionError(f"Proporsi kelas {split_name} menyimpang >2 poin persentase dari 50/50: {ratio:.4f}")
split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(OUTPUT_DIR / "tables" / "split_summary.csv", index=False)
# Simpan manifest kolom yang diminta protokol (path, label, slide_id, group_id, split).
manifest[["path", "label", "slide_id", "group_id", "split"]].to_csv(manifest_path, index=False)
print("Manifest:", manifest_path)

## 3. Definisi konfigurasi dan arsitektur CNN

Urutan setiap blok: `Conv2D → [BatchNorm] → ReLU → MaxPool → [Dropout 0,25]`. Setelah tiga blok: `GlobalAveragePooling → Dense(256, ReLU) → [Dropout 0,5] → Dense(1, sigmoid)`. BN hanya dipasang setelah Conv2D; tidak ada BN pada Dense. Bias Conv2D tetap aktif. Optimizer Adam dan loss binary crossentropy sama pada semua run.

In [ ]:
CONFIGS = {
    "A": {"bn": False, "dropout": False, "lr_mode": "constant", "phase": 1},
    "B": {"bn": True,  "dropout": False, "lr_mode": "constant", "phase": 1},
    "C": {"bn": False, "dropout": True,  "lr_mode": "constant", "phase": 1},
    "D": {"bn": True,  "dropout": True,  "lr_mode": "constant", "phase": 1},
    "E": {"bn": True,  "dropout": True,  "lr_mode": "plateau",  "phase": 1},
    "F": {"bn": True,  "dropout": True,  "lr_mode": "cosine",   "phase": 1},
    "G": {"bn": False, "dropout": False, "lr_mode": "plateau",  "phase": 2},
    "H": {"bn": False, "dropout": False, "lr_mode": "cosine",   "phase": 2},
    "I": {"bn": True,  "dropout": False, "lr_mode": "plateau",  "phase": 3},
    "J": {"bn": True,  "dropout": False, "lr_mode": "cosine",   "phase": 3},
    "K": {"bn": False, "dropout": True,  "lr_mode": "plateau",  "phase": 3},
    "L": {"bn": False, "dropout": True,  "lr_mode": "cosine",   "phase": 3},
}


def build_model(use_bn: bool, use_dropout: bool, image_size: int = IMAGE_SIZE):
    inputs = tf.keras.Input(shape=(image_size, image_size, 3), name="image")
    x = inputs
    for filters in [32, 64, 128]:
        x = tf.keras.layers.Conv2D(
            filters=filters, kernel_size=(5, 5), padding="same", use_bias=True,
            kernel_initializer="he_normal", bias_initializer="zeros",
        )(x)
        if use_bn:
            x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)
        if use_dropout:
            x = tf.keras.layers.Dropout(DROPOUT_CONV)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu", kernel_initializer="he_normal")(x)
    if use_dropout:
        x = tf.keras.layers.Dropout(DROPOUT_DENSE)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="parasitized_probability")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

# Uji jumlah parameter sesuai tabel protokol.
model_A_check = build_model(False, False)
model_B_check = build_model(True, False)
print("Parameter A:", model_A_check.count_params(), "(target 291905)")
print("Parameter B:", model_B_check.count_params(), "(target 292801)")
assert model_A_check.count_params() == 291905, "Parameter baseline tidak cocok dengan protokol."
assert model_B_check.count_params() == 292801, "Parameter model dengan BN tidak cocok dengan protokol."
model_A_check.summary()
del model_A_check, model_B_check
tf.keras.backend.clear_session()

## 4. Pipeline input (tanpa augmentasi)

Citra dibaca dari path, didekode sebagai RGB, di-resize bilinear, lalu dinormalisasi dengan pembagian 255. Data train diacak dengan seed run yang bersangkutan; validation dan test tidak diacak. Tidak ada augmentasi pada split mana pun.

In [ ]:
def decode_resize_normalize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], method="bilinear", antialias=False)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.float32)


def make_dataset(frame, training=False, seed=1):
    paths = frame.path.astype(str).values
    labels = frame.label.astype(np.float32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)
    if training:
        ds = ds.shuffle(buffer_size=len(frame), seed=int(seed), reshuffle_each_iteration=True)
    ds = ds.map(decode_resize_normalize, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

train_frame = sets["train"].reset_index(drop=True)
val_frame = sets["validation"].reset_index(drop=True)
print("Jumlah batch train:", int(np.ceil(len(train_frame) / BATCH_SIZE)))
print("Jumlah batch validation:", int(np.ceil(len(val_frame) / BATCH_SIZE)))

## 4A. Uji kewarasan: baseline harus dapat menghafal ±200 citra

Sesuai checklist protokol, baseline A dilatih pada subset **200 citra dari train saja** (100 per kelas). Uji ini bukan bagian dari hasil penelitian dan tidak menggunakan validation/test. Akurasi train yang mendekati 100% menjadi pemeriksaan bahwa pipeline dan model dapat belajar. Hasil disimpan agar tidak perlu diulang pada setiap eksekusi notebook.

In [ ]:
SANITY_CHECK_IMAGES_PER_CLASS = 100
SANITY_CHECK_EPOCHS = 200
SANITY_CHECK_THRESHOLD = 0.95
sanity_path = OUTPUT_DIR / "sanity_check_result.json"

# Helper didefinisikan di sini agar uji kewarasan dapat dijalankan sebelum pilot.
def set_run_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

def make_optimizer(learning_rate):
    return tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999)

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)

if sanity_path.exists():
    sanity_result = json.loads(sanity_path.read_text(encoding="utf-8"))
    if sanity_result.get("image_size") != IMAGE_SIZE or sanity_result.get("epochs") != SANITY_CHECK_EPOCHS:
        sanity_result = None
else:
    sanity_result = None

if sanity_result is None:
    sanity_parts = []
    for cls in [0, 1]:
        part = train_frame.loc[train_frame.label == cls].sample(
            n=SANITY_CHECK_IMAGES_PER_CLASS, random_state=2026
        )
        sanity_parts.append(part)
    sanity_frame = pd.concat(sanity_parts, ignore_index=True).sample(frac=1, random_state=2026).reset_index(drop=True)

    tf.keras.backend.clear_session()
    gc.collect()
    set_run_seed(2026)
    sanity_model = build_model(use_bn=False, use_dropout=False)
    sanity_model.compile(
        optimizer=make_optimizer(1e-3),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    sanity_history = sanity_model.fit(
        make_dataset(sanity_frame, training=True, seed=2026),
        epochs=SANITY_CHECK_EPOCHS,
        verbose=2,
    )
    sanity_train_accuracy = float(sanity_history.history["accuracy"][-1])
    sanity_result = {
        "images": int(len(sanity_frame)),
        "images_per_class": SANITY_CHECK_IMAGES_PER_CLASS,
        "epochs": SANITY_CHECK_EPOCHS,
        "train_accuracy_last_epoch": sanity_train_accuracy,
        "threshold": SANITY_CHECK_THRESHOLD,
        "image_size": IMAGE_SIZE,
        "passed": bool(sanity_train_accuracy >= SANITY_CHECK_THRESHOLD),
    }
    save_json(sanity_path, sanity_result)
    del sanity_model, sanity_history
    tf.keras.backend.clear_session()
    gc.collect()

print("Hasil sanity check:", sanity_result)
if not sanity_result["passed"]:
    raise RuntimeError(
        f"Sanity check belum mencapai akurasi train {SANITY_CHECK_THRESHOLD:.0%}. "
        "Periksa pipeline/model sebelum melanjutkan eksperimen utama."
    )

## 5. Pilot learning rate otomatis

Pilot hanya melatih konfigurasi A (tanpa BN dan Dropout), satu seed (seed 1), selama 15 epoch untuk masing-masing LR `3e-4` dan `1e-3`. LR dengan nilai **minimum validation loss** paling rendah dipilih dan disimpan ke `pilot/lr_selection.json`. Test set tidak digunakan. Jika notebook dijalankan ulang, hasil pilot yang sudah selesai digunakan kecuali `FORCE_RERUN_PILOT=True`.

In [ ]:
def set_run_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def make_optimizer(learning_rate):
    return tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999)


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)


class EpochLearningRateRecorder(tf.keras.callbacks.Callback):
    """Catat LR aktual yang digunakan pada awal setiap epoch."""
    def on_train_begin(self, logs=None):
        self.learning_rates = []
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        # LearningRateSchedule (callable, tanpa .numpy()): evaluasi pada iterasi saat ini.
        if callable(lr) and not hasattr(lr, "numpy"):
            lr = lr(self.model.optimizer.iterations)
        # Keras 3: Variable memiliki .numpy(); Keras 2/tf.keras: tf.Variable juga. Tanpa tf.keras.backend.get_value.
        value = float(np.asarray(lr.numpy() if hasattr(lr, "numpy") else lr))
        self.learning_rates.append(value)


def run_pilot_trial(lr):
    name = f"lr_{lr:.0e}"
    trial_dir = OUTPUT_DIR / "pilot" / name
    trial_dir.mkdir(parents=True, exist_ok=True)
    weights_path = trial_dir / "best.weights.h5"
    history_path = trial_dir / "history.csv"
    result_path = trial_dir / "result.json"
    if result_path.exists() and weights_path.exists() and not FORCE_RERUN_PILOT:
        result = json.loads(result_path.read_text(encoding="utf-8"))
        if result.get("image_size") == IMAGE_SIZE and result.get("dataset_fingerprint") == dataset_fingerprint:
            print("Pilot sudah ada, memakai hasil:", name)
            return result

    tf.keras.backend.clear_session()
    gc.collect()
    set_run_seed(1)
    model = build_model(use_bn=False, use_dropout=False)
    model.compile(optimizer=make_optimizer(lr), loss="binary_crossentropy", metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")])
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(weights_path), monitor="val_loss", mode="min", save_best_only=True, save_weights_only=True, verbose=0),
    ]
    lr_recorder = EpochLearningRateRecorder()
    callbacks.append(lr_recorder)
    start = time.perf_counter()
    hist = model.fit(
        make_dataset(train_frame, training=True, seed=1),
        validation_data=make_dataset(val_frame, training=False),
        epochs=PILOT_EPOCHS, callbacks=callbacks, verbose=2,
    )
    duration = time.perf_counter() - start
    h = pd.DataFrame(hist.history)
    h.insert(0, "epoch", np.arange(1, len(h) + 1))
    h["learning_rate"] = lr_recorder.learning_rates[:len(h)]
    h.to_csv(history_path, index=False)
    best_idx = int(np.argmin(h["val_loss"].values))
    result = {
        "learning_rate": float(lr),
        "image_size": IMAGE_SIZE,
        "dataset_fingerprint": dataset_fingerprint,
        "best_val_loss": float(h.loc[best_idx, "val_loss"]),
        "best_epoch": int(h.loc[best_idx, "epoch"]),
        "train_loss_at_best_epoch": float(h.loc[best_idx, "loss"]),
        "epochs_completed": int(len(h)),
        "duration_seconds": float(duration),
        "weights": str(weights_path),
    }
    save_json(result_path, result)
    print(f"Pilot {name}: min val_loss={result['best_val_loss']:.6f} pada epoch {result['best_epoch']}")
    del model, hist
    tf.keras.backend.clear_session()
    gc.collect()
    return result

pilot_selection_path = OUTPUT_DIR / "pilot" / "lr_selection.json"
if pilot_selection_path.exists() and not FORCE_RERUN_PILOT:
    pilot_selection = json.loads(pilot_selection_path.read_text(encoding="utf-8"))
    if pilot_selection.get("image_size") == IMAGE_SIZE and pilot_selection.get("dataset_fingerprint") == dataset_fingerprint:
        print("Memakai LR pilot yang sudah dipilih:", pilot_selection)
    else:
        pilot_selection = None
else:
    pilot_selection = None

if pilot_selection is None:
    pilot_results = [run_pilot_trial(lr) for lr in PILOT_LR_CANDIDATES]
    pilot_results = sorted(pilot_results, key=lambda x: x["best_val_loss"])
    pilot_selection = {
        "selected_learning_rate": pilot_results[0]["learning_rate"],
        "image_size": IMAGE_SIZE,
        "dataset_fingerprint": dataset_fingerprint,
        "selection_rule": "minimum validation loss terendah pada pilot baseline A",
        "pilot_epochs": PILOT_EPOCHS,
        "seed": 1,
        "candidates": pilot_results,
        "selected_at": datetime.now().isoformat(timespec="seconds"),
    }
    save_json(pilot_selection_path, pilot_selection)

INITIAL_LR = float(pilot_selection["selected_learning_rate"])
print(f"\nLR TERPILIH UNTUK SEMUA KONFIGURASI: {INITIAL_LR:g}")

## 6. Penjadwalan fase dan pelatihan A–L

Urutan run mengikuti prioritas protokol:
- **Fase 1:** A–F, seed 1–3 (seed 1 lebih dahulu untuk A–F sebagai smoke test awal).
- **Fase 2:** G–H, seed 1–3.
- **Fase 3:** I–L, seed 1–3.
- **Fase 4:** seed 4 lalu 5 untuk semua konfigurasi yang masuk.

Konfigurasi plateau memakai `ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6)`. Cosine decay dihitung per epoch dari LR awal ke `1e-6` selama 40 epoch. Tidak ada early stopping. Setiap run menyimpan config, riwayat epoch, checkpoint validation loss terbaik, dan metadata waktu.

In [ ]:
# Urutan fase: seed 1 untuk konfigurasi fase lebih dulu, lalu seed 2,3.
PHASE_CONFIGS = {
    1: ["A", "B", "C", "D", "E", "F"],
    2: ["G", "H"],
    3: ["I", "J", "K", "L"],
}
RUN_PLAN = []
for phase in [1, 2, 3]:
    if phase > MAX_PHASE_TO_RUN:
        continue
    for seed in [1, 2, 3]:
        for config_name in PHASE_CONFIGS[phase]:
            RUN_PLAN.append((phase, config_name, seed))
if MAX_PHASE_TO_RUN >= 4:
    for seed in [4, 5]:
        for config_name in CONFIGS.keys():
            RUN_PLAN.append((4, config_name, seed))

print("Total run yang dijadwalkan:", len(RUN_PLAN))
print("Konfigurasi yang dijadwalkan:", sorted({x[1] for x in RUN_PLAN}))


def cosine_lr_for_epoch(epoch_index, total_epochs=EPOCHS, initial_lr=INITIAL_LR, min_lr=MIN_LR):
    # Epoch pertama = initial LR; epoch terakhir = min LR.
    if total_epochs <= 1:
        return float(min_lr)
    progress = min(max(epoch_index / (total_epochs - 1), 0.0), 1.0)
    return float(min_lr + 0.5 * (initial_lr - min_lr) * (1.0 + np.cos(np.pi * progress)))


def make_training_callbacks(config_name, seed, run_dir):
    cfg = CONFIGS[config_name]
    best_path = run_dir / "best.weights.h5"
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_loss", mode="min", save_best_only=True, save_weights_only=True, verbose=0),
    ]
    if cfg["lr_mode"] == "plateau":
        callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=MIN_LR, verbose=1
        ))
    elif cfg["lr_mode"] == "cosine":
        callbacks.append(tf.keras.callbacks.LearningRateScheduler(
            lambda epoch, lr: cosine_lr_for_epoch(epoch), verbose=0
        ))
    lr_recorder = EpochLearningRateRecorder()
    callbacks.append(lr_recorder)
    return callbacks, lr_recorder


def train_one_run(config_name, seed, phase):
    run_id = f"{config_name}_seed_{seed}"
    run_dir = OUTPUT_DIR / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    status_path = run_dir / "training_metadata.json"
    weights_path = run_dir / "best.weights.h5"
    history_path = run_dir / "history.csv"
    if status_path.exists() and weights_path.exists() and history_path.exists() and not FORCE_RERUN_COMPLETED_RUNS:
        old = json.loads(status_path.read_text(encoding="utf-8"))
        compatible = (
            old.get("status") == "completed"
            and old.get("initial_learning_rate") == INITIAL_LR
            and old.get("image_size") == IMAGE_SIZE
            and old.get("batch_size") == BATCH_SIZE
            and old.get("epochs") == EPOCHS
            and old.get("dataset_fingerprint") == dataset_fingerprint
            and old.get("config") == config_name
            and int(old.get("seed", -1)) == int(seed)
        )
        if compatible:
            print(f"[SKIP] {run_id} sudah selesai dan kompatibel dengan pengaturan saat ini.")
            return old

    tf.keras.backend.clear_session()
    gc.collect()
    set_run_seed(seed)
    cfg = CONFIGS[config_name]
    model = build_model(cfg["bn"], cfg["dropout"])
    model.compile(
        optimizer=make_optimizer(INITIAL_LR),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    params = int(model.count_params())
    callbacks, lr_recorder = make_training_callbacks(config_name, seed, run_dir)
    config_record = {
        "config": config_name, "phase": phase, "seed": seed,
        "batch_normalization": bool(cfg["bn"]), "dropout": bool(cfg["dropout"]),
        "lr_mode": cfg["lr_mode"], "initial_learning_rate": INITIAL_LR,
        "min_learning_rate": MIN_LR, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "image_size": IMAGE_SIZE, "dataset_fingerprint": dataset_fingerprint, "dropout_conv": DROPOUT_CONV,
        "dropout_dense": DROPOUT_DENSE, "threshold": THRESHOLD,
        "optimizer": "Adam(beta_1=0.9, beta_2=0.999)", "loss": "binary_crossentropy",
        "augmentation": "none", "precision": "float32", "parameters": params,
    }
    save_json(run_dir / "config.json", config_record)

    print(f"\n[TRAIN] {run_id} | BN={cfg['bn']} Dropout={cfg['dropout']} LR={cfg['lr_mode']} | params={params:,}")
    start = time.perf_counter()
    hist = model.fit(
        make_dataset(train_frame, training=True, seed=seed),
        validation_data=make_dataset(val_frame, training=False),
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=2,
    )
    duration = time.perf_counter() - start

    history_df = pd.DataFrame(hist.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_df["learning_rate"] = lr_recorder.learning_rates[:len(history_df)]
    history_df.to_csv(history_path, index=False)
    if not weights_path.exists():
        raise RuntimeError(f"Checkpoint terbaik tidak ditemukan untuk {run_id}.")

    best_idx = int(np.argmin(history_df["val_loss"].values))
    min_val_loss = float(history_df.loc[best_idx, "val_loss"])
    train_loss_at_best = float(history_df.loc[best_idx, "loss"])
    convergence_epochs = np.where(history_df["val_loss"].values <= 1.05 * min_val_loss)[0]
    convergence_epoch = int(convergence_epochs[0] + 1) if len(convergence_epochs) else None
    metadata = {
        "status": "completed", "run_id": run_id, **config_record,
        "epochs_completed": int(len(history_df)),
        "best_epoch": int(history_df.loc[best_idx, "epoch"]),
        "best_val_loss": min_val_loss,
        "train_loss_at_best_epoch": train_loss_at_best,
        "val_minus_train_loss_at_best": float(min_val_loss - train_loss_at_best),
        "convergence_epoch": convergence_epoch,
        "training_duration_seconds": float(duration),
        "seconds_per_epoch": float(duration / max(len(history_df), 1)),
        "completed_at": datetime.now().isoformat(timespec="seconds"),
    }
    save_json(status_path, metadata)
    print(f"[DONE] {run_id}: best val_loss={min_val_loss:.6f} epoch={metadata['best_epoch']} | {duration/60:.1f} menit")
    del model, hist, history_df
    tf.keras.backend.clear_session()
    gc.collect()
    return metadata

training_metadata = []
for phase, config_name, seed in RUN_PLAN:
    training_metadata.append(train_one_run(config_name, seed, phase))

print("\nTahap training selesai untuk seluruh run yang dijadwalkan.")

## 7. Evaluasi test final

Sel ini baru membuat pipeline test setelah loop training selesai. Tiap checkpoint terbaik dievaluasi satu kali pada test set (threshold 0,5). Prediksi per citra dan metrik per run disimpan agar evaluasi tidak diulang ketika notebook dijalankan lagi.

In [ ]:
def make_test_dataset(frame):
    paths = frame.path.astype(str).values
    labels = frame.label.astype(np.float32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    options = tf.data.Options(); options.experimental_deterministic = True
    ds = ds.with_options(options)
    ds = ds.map(decode_resize_normalize, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    return ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)


def evaluate_one_run(config_name, seed):
    run_id = f"{config_name}_seed_{seed}"
    run_dir = OUTPUT_DIR / "runs" / run_id
    metrics_path = run_dir / "test_metrics.json"
    predictions_path = run_dir / "test_predictions.csv"
    weights_path = run_dir / "best.weights.h5"
    if metrics_path.exists() and predictions_path.exists() and not FORCE_RERUN_COMPLETED_RUNS:
        print(f"[TEST SKIP] {run_id} sudah dievaluasi.")
        return json.loads(metrics_path.read_text(encoding="utf-8"))
    if not weights_path.exists():
        print(f"[TEST SKIP] checkpoint tidak ada: {run_id}")
        return None

    cfg = CONFIGS[config_name]
    tf.keras.backend.clear_session()
    model = build_model(cfg["bn"], cfg["dropout"])
    model.load_weights(weights_path)
    # Satu forward pass test; metrik dihitung dari probabilitas yang sama.
    test_ds = make_test_dataset(test_frame)
    y_true = test_frame.label.to_numpy(dtype=np.int32)
    y_prob = model.predict(test_ds.map(lambda x, y: x), verbose=0).reshape(-1)
    y_pred = (y_prob >= THRESHOLD).astype(np.int32)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    eps = 1e-7
    clipped = np.clip(y_prob, eps, 1 - eps)
    test_loss = float(np.mean(-(y_true * np.log(clipped) + (1 - y_true) * np.log(1 - clipped))))
    try:
        auc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        auc = float("nan")
    metrics = {
        "run_id": run_id, "config": config_name, "seed": int(seed),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_parasitized": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall_parasitized": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_parasitized": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc_roc": auc, "test_loss": test_loss,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "test_images": int(len(y_true)), "threshold": THRESHOLD,
    }
    pred_df = pd.DataFrame({
        "path": test_frame.path.values,
        "y_true": y_true,
        "y_probability_parasitized": y_prob,
        "y_pred": y_pred,
        "error_type": np.where((y_true == 0) & (y_pred == 1), "FP",
                         np.where((y_true == 1) & (y_pred == 0), "FN", "correct")),
    })
    pred_df.to_csv(predictions_path, index=False)
    save_json(metrics_path, metrics)
    print(f"[TEST DONE] {run_id}: F1={metrics['f1_parasitized']:.4f} | AUC={metrics['auc_roc']:.4f}")
    del model, test_ds
    tf.keras.backend.clear_session()
    gc.collect()
    return metrics


completed_pairs = []
for run_dir in sorted((OUTPUT_DIR / "runs").glob("*_seed_*")):
    status_file = run_dir / "training_metadata.json"
    if status_file.exists():
        status = json.loads(status_file.read_text(encoding="utf-8"))
        if status.get("status") == "completed":
            completed_pairs.append((status["config"], int(status["seed"])))

# Jangan membuka/evaluasi test set sebelum SEMUA 12 konfigurasi × 5 seed selesai.
expected_pairs = {(cfg, seed) for cfg in CONFIGS for seed in SEEDS}
completed_set = set(completed_pairs)
missing_pairs = sorted(expected_pairs - completed_set)
if MAX_PHASE_TO_RUN != 4 or missing_pairs:
    raise RuntimeError(
        f"Evaluasi test ditahan agar tidak melanggar urutan protokol. Run belum lengkap: {len(missing_pairs)}. "
        "Selesaikan seluruh fase (MAX_PHASE_TO_RUN=4), lalu jalankan ulang notebook. "
        "Test set belum dibaca oleh model."
    )

test_frame = sets["test"].reset_index(drop=True)

all_test_metrics = []
for config_name, seed in completed_pairs:
    result = evaluate_one_run(config_name, seed)
    if result is not None:
        all_test_metrics.append(result)

if not all_test_metrics:
    raise RuntimeError("Tidak ada run lengkap yang dapat dievaluasi. Periksa status training.")

metrics_df = pd.DataFrame(all_test_metrics).sort_values(["config", "seed"])
metrics_df.to_csv(OUTPUT_DIR / "tables" / "test_metrics_per_run.csv", index=False)
print("Jumlah run yang sudah dievaluasi:", len(metrics_df))
metrics_df.head()

## 8. Ringkasan per konfigurasi dan perbandingan berpasangan

Metrik dilaporkan sebagai mean ± SD lintas seed. F1 kelas `parasitized` adalah metrik utama. Aturan interpretasi protokol diterapkan secara mekanis: rentang mean ± SD tidak tumpang tindih dan arah selisih sama pada minimal 4 dari 5 seed (atau semua seed bila n=3). Dengan n=4, notebook menggunakan minimal 4 arah konsisten sebagai pendekatan konservatif.

In [ ]:
metric_cols = ["accuracy", "precision_parasitized", "recall_parasitized", "f1_parasitized", "auc_roc", "test_loss"]
summary_rows = []
for config_name, group in metrics_df.groupby("config"):
    row = {"config": config_name, "n_seeds": int(group.seed.nunique())}
    for metric in metric_cols:
        row[f"{metric}_mean"] = float(group[metric].mean())
        row[f"{metric}_sd"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
    row["confusion_TN_mean"] = float(group.tn.mean())
    row["confusion_FP_mean"] = float(group.fp.mean())
    row["confusion_FN_mean"] = float(group.fn.mean())
    row["confusion_TP_mean"] = float(group.tp.mean())
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).sort_values("config")
summary_df.to_csv(OUTPUT_DIR / "tables" / "summary_by_configuration.csv", index=False)

# Tambahkan biaya pelatihan dan parameter dari metadata training.
metadata_rows = []
for cfg, seed in completed_pairs:
    p = OUTPUT_DIR / "runs" / f"{cfg}_seed_{seed}" / "training_metadata.json"
    m = json.loads(p.read_text(encoding="utf-8"))
    metadata_rows.append({"config": cfg, "seed": seed, "parameters": m["parameters"],
                          "seconds_per_epoch": m["seconds_per_epoch"],
                          "training_duration_seconds": m["training_duration_seconds"],
                          "best_epoch": m["best_epoch"], "convergence_epoch": m["convergence_epoch"],
                          "val_minus_train_loss_at_best": m["val_minus_train_loss_at_best"]})
metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(OUTPUT_DIR / "tables" / "training_cost_per_run.csv", index=False)

summary_with_cost = summary_df.merge(
    metadata_df.groupby("config").agg(
        parameters=("parameters", "first"),
        seconds_per_epoch_mean=("seconds_per_epoch", "mean"),
        seconds_per_epoch_sd=("seconds_per_epoch", "std"),
        convergence_epoch_mean=("convergence_epoch", "mean"),
        convergence_epoch_sd=("convergence_epoch", "std"),
        train_val_gap_mean=("val_minus_train_loss_at_best", "mean"),
        train_val_gap_sd=("val_minus_train_loss_at_best", "std"),
    ).reset_index(), on="config", how="left")
summary_with_cost.to_csv(OUTPUT_DIR / "tables" / "summary_by_configuration_with_cost.csv", index=False)

# Perbandingan terencana: nilai selisih adalah F1(konfigurasi kanan) - F1(konfigurasi kiri).
planned_pairs = [
    ("A", "B", "Efek BN pada LR konstan"),
    ("A", "C", "Efek Dropout pada LR konstan"),
    ("A", "D", "Efek gabungan BN+Dropout pada LR konstan"),
    ("B", "D", "Nilai tambah Dropout ketika BN aktif (LR konstan)"),
    ("C", "D", "Nilai tambah BN ketika Dropout aktif (LR konstan)"),
    ("A", "G", "ReduceLROnPlateau pada baseline"),
    ("A", "H", "Cosine pada baseline"),
    ("B", "I", "ReduceLROnPlateau pada BN saja"),
    ("B", "J", "Cosine pada BN saja"),
    ("C", "K", "ReduceLROnPlateau pada Dropout saja"),
    ("C", "L", "Cosine pada Dropout saja"),
    ("D", "E", "ReduceLROnPlateau di atas BN+Dropout"),
    ("D", "F", "Cosine di atas BN+Dropout"),
    ("G", "H", "ReduceLROnPlateau vs Cosine pada baseline"),
    ("I", "J", "ReduceLROnPlateau vs Cosine pada BN saja"),
    ("K", "L", "ReduceLROnPlateau vs Cosine pada Dropout saja"),
    ("E", "F", "ReduceLROnPlateau vs Cosine pada BN+Dropout"),
]

pair_rows = []
for left, right, description in planned_pairs:
    left_df = metrics_df.loc[metrics_df.config == left, ["seed", "f1_parasitized"]].rename(columns={"f1_parasitized": "f1_left"})
    right_df = metrics_df.loc[metrics_df.config == right, ["seed", "f1_parasitized"]].rename(columns={"f1_parasitized": "f1_right"})
    paired = left_df.merge(right_df, on="seed", how="inner")
    if paired.empty:
        continue
    diffs = (paired.f1_right - paired.f1_left).to_numpy(dtype=float)
    mean_diff = float(np.mean(diffs))
    sd_diff = float(np.std(diffs, ddof=1)) if len(diffs) > 1 else 0.0
    n = len(diffs)
    consistent = int(np.sum(diffs >= 0)) if mean_diff >= 0 else int(np.sum(diffs <= 0))
    required_consistency = n if n == 3 else 4
    lrow = summary_df.loc[summary_df.config == left]
    rrow = summary_df.loc[summary_df.config == right]
    nonoverlap = False
    if not lrow.empty and not rrow.empty:
        lm, ls = float(lrow.iloc[0].f1_parasitized_mean), float(lrow.iloc[0].f1_parasitized_sd)
        rm, rs = float(rrow.iloc[0].f1_parasitized_mean), float(rrow.iloc[0].f1_parasitized_sd)
        nonoverlap = (lm + ls < rm - rs) or (rm + rs < lm - ls)
    claim_rule = bool(nonoverlap and consistent >= required_consistency and mean_diff != 0)
    pair_rows.append({
        "comparison": f"{right}-{left}", "description": description, "n_paired_seeds": n,
        "mean_F1_difference_right_minus_left": mean_diff,
        "sd_of_paired_difference": sd_diff,
        "direction_consistent_seeds": consistent,
        "required_consistent_seeds": required_consistency,
        "mean_plus_minus_SD_nonoverlap": nonoverlap,
        "protocol_rule_met": claim_rule,
    })
pairwise_df = pd.DataFrame(pair_rows)
pairwise_df.to_csv(OUTPUT_DIR / "tables" / "planned_pairwise_comparisons.csv", index=False)

print("Ringkasan konfigurasi:")
display(summary_with_cost[["config", "n_seeds", "f1_parasitized_mean", "f1_parasitized_sd", "accuracy_mean", "recall_parasitized_mean", "parameters", "seconds_per_epoch_mean"]])
print("Perbandingan berpasangan:")
display(pairwise_df)

## 9. ANOVA tiga arah (hanya desain lengkap)

ANOVA dijalankan hanya bila seluruh A–L tersedia dan jumlah seed per konfigurasi sama (minimal 3), sesuai protokol. Faktor: BN (ya/tidak), Dropout (ya/tidak), dan LR mode (constant/plateau/cosine). Bila desain belum lengkap atau tidak seimbang, ANOVA dilewati dan hasil hanya diringkas secara deskriptif.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

required_configs = set(CONFIGS.keys())
observed_configs = set(metrics_df.config.unique())
seed_counts = metrics_df.groupby("config").seed.nunique()
run_anova = (observed_configs == required_configs and len(seed_counts) == 12 and seed_counts.nunique() == 1 and int(seed_counts.iloc[0]) >= 3)

if run_anova:
    anova_data = metrics_df.copy()
    anova_data["BN"] = anova_data.config.map(lambda c: "yes" if CONFIGS[c]["bn"] else "no")
    anova_data["Dropout"] = anova_data.config.map(lambda c: "yes" if CONFIGS[c]["dropout"] else "no")
    anova_data["LR_mode"] = anova_data.config.map(lambda c: CONFIGS[c]["lr_mode"])
    model = smf.ols("f1_parasitized ~ C(BN) * C(Dropout) * C(LR_mode)", data=anova_data).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    anova_table.to_csv(OUTPUT_DIR / "tables" / "anova_three_way_F1.csv")
    (OUTPUT_DIR / "tables" / "anova_three_way_F1.txt").write_text(anova_table.to_string(), encoding="utf-8")
    print("ANOVA tiga arah F1 (Type II):")
    display(anova_table)
else:
    print("ANOVA tiga arah dilewati: A-L belum lengkap / jumlah seed tidak seimbang / seed < 3.")

## 10. Kurva training dan analisis kesalahan FP/FN

Kurva mean ± SD dibuat untuk konfigurasi kunci A, D, E, F, G, H yang tercantum dalam protokol. Untuk analisis kesalahan, konfigurasi dengan mean F1 tertinggi dipilih secara deskriptif. Contoh citra FP/FN diambil dari seed pada konfigurasi tersebut yang F1-nya paling dekat ke mean konfigurasi (bukan dari seed terbaik), lalu diekspor sebagai berkas gambar dan CSV. Grad-CAM tidak dibuat karena protokol secara eksplisit memutuskan untuk tidak menambahkannya.

In [ ]:
PLOT_CONFIGS = ["A", "D", "E", "F", "G", "H"]

# Gabungkan riwayat yang benar-benar tersedia.
history_map = {}
for cfg in sorted(metrics_df.config.unique()):
    frames = []
    for seed in sorted(metrics_df.loc[metrics_df.config == cfg, "seed"].unique()):
        hp = OUTPUT_DIR / "runs" / f"{cfg}_seed_{seed}" / "history.csv"
        if hp.exists():
            h = pd.read_csv(hp)
            h["seed"] = int(seed)
            frames.append(h)
    if frames:
        history_map[cfg] = pd.concat(frames, ignore_index=True)

for metric, ylabel, filename, title in [
    ("loss", "Loss", "kurva_loss_konfigurasi.png", "Kurva loss training dan validation"),
    ("accuracy", "Accuracy", "kurva_akurasi_konfigurasi.png", "Kurva accuracy training dan validation"),
]:
    plt.figure(figsize=(11, 7))
    plotted = False
    for cfg in PLOT_CONFIGS:
        if cfg not in history_map:
            continue
        h = history_map[cfg]
        for split_prefix, style in [("", "-"), ("val_", "--")]:
            col = f"{split_prefix}{metric}"
            if col not in h.columns:
                continue
            grouped = h.groupby("epoch")[col]
            mean = grouped.mean()
            sd = grouped.std(ddof=1).fillna(0)
            x = mean.index.to_numpy()
            y = mean.to_numpy()
            plt.plot(x, y, linestyle=style, label=f"{cfg} {'val' if split_prefix else 'train'}")
            plt.fill_between(x, y - sd.to_numpy(), y + sd.to_numpy(), alpha=0.10)
            plotted = True
    if plotted:
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.title(title + " (mean ± SD)")
        plt.legend(ncol=2, fontsize=8)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "plots" / filename, dpi=180)
    plt.show()
    plt.close()

# Tentukan konfigurasi teratas menurut mean F1.
best_config = summary_df.sort_values("f1_parasitized_mean", ascending=False).iloc[0]["config"]
best_mean_f1 = float(summary_df.loc[summary_df.config == best_config, "f1_parasitized_mean"].iloc[0])
seed_scores = metrics_df.loc[metrics_df.config == best_config, ["seed", "f1_parasitized"]].copy()
seed_scores["distance_to_mean"] = (seed_scores.f1_parasitized - best_mean_f1).abs()
representative_seed = int(seed_scores.sort_values("distance_to_mean").iloc[0].seed)
print(f"Konfigurasi mean F1 tertinggi: {best_config} (mean F1={best_mean_f1:.4f}); seed representatif={representative_seed}")

rep_pred_path = OUTPUT_DIR / "runs" / f"{best_config}_seed_{representative_seed}" / "test_predictions.csv"
rep_predictions = pd.read_csv(rep_pred_path)
error_root = OUTPUT_DIR / "error_examples" / f"{best_config}_seed_{representative_seed}"
for kind in ["FP", "FN"]:
    (error_root / kind).mkdir(parents=True, exist_ok=True)

error_manifest = []
MAX_EXAMPLES_PER_TYPE = 16
for kind in ["FP", "FN"]:
    subset = rep_predictions.loc[rep_predictions.error_type == kind].copy()
    # Tampilkan prediksi paling yakin untuk inspeksi kesalahan.
    if kind == "FP":
        subset["confidence"] = subset.y_probability_parasitized
        subset = subset.sort_values("confidence", ascending=False)
    else:
        subset["confidence"] = 1.0 - subset.y_probability_parasitized
        subset = subset.sort_values("confidence", ascending=False)
    subset = subset.head(MAX_EXAMPLES_PER_TYPE)
    for idx, row in enumerate(subset.itertuples(index=False), start=1):
        src = Path(row.path)
        out = error_root / kind / f"{kind}_{idx:02d}_{src.stem}.png"
        with Image.open(src) as im:
            im.convert("RGB").save(out)
        error_manifest.append({"type": kind, "source_path": str(src), "saved_path": str(out),
                               "y_true": int(row.y_true), "probability": float(row.y_probability_parasitized)})

pd.DataFrame(error_manifest).to_csv(error_root / "error_examples_manifest.csv", index=False)
print("Contoh FP:", int((rep_predictions.error_type == "FP").sum()), "| contoh FN:", int((rep_predictions.error_type == "FN").sum()))
print("Contoh gambar disimpan di:", error_root)

## 11. Catatan reproducibility dan laporan akhir

Notebook menyimpan versi pustaka, perangkat, split manifest, pengaturan eksperimen, metadata tiap run, dan tabel hasil. Periksa `summary_by_configuration_with_cost.csv`, `planned_pairwise_comparisons.csv`, `anova_three_way_F1.csv` (bila memenuhi syarat), dan folder `plots/`.

In [ ]:
import sklearn
import PIL
import matplotlib
import statsmodels

environment_info = {
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "pillow": PIL.__version__,
    "matplotlib": matplotlib.__version__,
    "statsmodels": statsmodels.__version__,
    "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")],
    "deterministic_ops_enabled": DETERMINISM_ENABLED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "pilot_epochs": PILOT_EPOCHS,
    "selected_initial_learning_rate": INITIAL_LR,
    "max_phase_to_run": MAX_PHASE_TO_RUN,
    "seeds": SEEDS,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
save_json(OUTPUT_DIR / "environment.json", environment_info)

print("=" * 72)
print("EKSPERIMEN SELESAI / STATUS HASIL")
print("=" * 72)
print("Dataset images:", len(df))
print("Split manifest:", manifest_path)
print("LR terpilih dari pilot:", INITIAL_LR)
print("Run yang telah dilatih:", len(completed_pairs))
print("Run yang telah dievaluasi:", len(metrics_df))
print("Konfigurasi yang dievaluasi:", sorted(metrics_df.config.unique()))
print("Folder seluruh hasil:", OUTPUT_DIR.resolve())
print("\nFile hasil utama:")
for p in [
    OUTPUT_DIR / "audit_summary.json",
    OUTPUT_DIR / "split_manifest.csv",
    OUTPUT_DIR / "pilot" / "lr_selection.json",
    OUTPUT_DIR / "tables" / "test_metrics_per_run.csv",
    OUTPUT_DIR / "tables" / "summary_by_configuration_with_cost.csv",
    OUTPUT_DIR / "tables" / "planned_pairwise_comparisons.csv",
    OUTPUT_DIR / "tables" / "anova_three_way_F1.csv",
    OUTPUT_DIR / "plots" / "kurva_loss_konfigurasi.png",
    OUTPUT_DIR / "plots" / "kurva_akurasi_konfigurasi.png",
]:
    print(("[ADA] " if p.exists() else "[TIDAK ADA / TIDAK BERLAKU] ") + str(p))

display(summary_with_cost.sort_values("f1_parasitized_mean", ascending=False))